In [ ]:
# ==============================
# STEP 1: IMPORT LIBRARIES
# ==============================
import pandas as pd
import numpy as np

# Visualization (optional but useful)
import matplotlib.pyplot as plt
import seaborn as sns

# Sklearn basics
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Metrics
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report

In [ ]:
# ==============================
# STEP 2: LOAD DATASET
# ==============================
df = pd.read_csv("diabetes_prediction_dataset.csv")

# Quick check
print(df.head())
print(df.info())
print(df.describe())

In [ ]:
# ==============================
# STEP 3: DATA CLEANING
# ==============================

# Convert categorical columns (if present)
df = pd.get_dummies(df, drop_first=True)

# Check missing values
print(df.isnull().sum())

# If needed:
df = df.dropna()  # simple approach for now

In [ ]:
# ==============================
# STEP 4: SPLIT FEATURES & TARGET
# ==============================

X = df.drop("diabetes", axis=1)   # target column
y = df["diabetes"]

print("Features shape:", X.shape)
print("Target shape:", y.shape)

In [ ]:
# ==============================
# STEP 5: TRAIN-TEST SPLIT
# ==============================

X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.2, 
    random_state=42,
    stratify=y   # important for imbalance
)

In [ ]:
# ==============================
# STEP 6: FEATURE SCALING
# ==============================

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
# ==============================
# STEP 7: EVALUATION FUNCTION (REUSABLE)
# ==============================

def evaluate_model(model, X_test, y_test):
    y_pred = model.predict(X_test)

    print("Accuracy:", accuracy_score(y_test, y_pred))
    print("Precision:", precision_score(y_test, y_pred))
    print("Recall:", recall_score(y_test, y_pred))
    print("F1 Score:", f1_score(y_test, y_pred))
    print("\nClassification Report:\n", classification_report(y_test, y_pred))

In [ ]:
# ==========================================
# MODELS
# ==========================================
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC

models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, class_weight='balanced'),
    "Decision Tree": DecisionTreeClassifier(max_depth=5),
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42),
    "SVM": SVC(probability=True),
    "Gradient Boosting": GradientBoostingClassifier(random_state=42)
}

import pandas as pd

results = []
trained_models = {}

for name, model in models.items():
    print(f"\n===== {name} =====")

    # Scale where needed
    if name in ["Logistic Regression", "SVM"]:
        model.fit(X_train_scaled, y_train)
        y_pred = model.predict(X_test_scaled)
    else:
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)

    metrics = evaluate(model, X_test_scaled if name in ["Logistic Regression", "SVM"] else X_test, y_test)

    print(metrics)

    results.append({
        "Model": name,
        **metrics
    })

    trained_models[name] = model

# Compare
results_df = pd.DataFrame(results).sort_values(by="F1", ascending=False)

print("\n=== BASELINE COMPARISON ===")
print(results_df)


In [ ]:
top_models = results_df.head(2)["Model"].values
print("Top Models:", top_models)

In [ ]:
from imblearn.over_sampling import SMOTE

sm = SMOTE(random_state=42)
X_train_res, y_train_res = sm.fit_resample(X_train, y_train)

print("Before SMOTE:\n", y_train.value_counts())
print("After SMOTE:\n", y_train_res.value_counts())

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train_res)
X_test_scaled = scaler.transform(X_test)


In [ ]:
import numpy as np
from sklearn.metrics import f1_score

final_results = {}

for name in top_models:
    model = models[name]

    print(f"\n===== {name} (After SMOTE) =====")

    # Train
    if name in ["Logistic Regression", "SVM"]:
        model.fit(X_train_scaled, y_train_res)
        probs = model.predict_proba(X_test_scaled)[:, 1]
    else:
        model.fit(X_train_res, y_train_res)
        probs = model.predict_proba(X_test)[:, 1]

    # Threshold tuning
    best_thresh = 0
    best_f1 = 0

    for t in np.arange(0.1, 0.9, 0.05):
        preds = (probs >= t).astype(int)
        f1 = f1_score(y_test, preds)

        if f1 > best_f1:
            best_f1 = f1
            best_thresh = t

    # Final prediction
    final_preds = (probs >= best_thresh).astype(int)

    metrics = {
        "Accuracy": accuracy_score(y_test, final_preds),
        "Precision": precision_score(y_test, final_preds),
        "Recall": recall_score(y_test, final_preds),
        "F1": f1_score(y_test, final_preds),
        "Threshold": best_thresh
    }

    print(f"Best Threshold: {best_thresh:.2f} | Best F1: {best_f1:.3f}")
    print(metrics)

    final_results[name] = {
        "model": model,
        **metrics
    }

In [ ]:
best_model_name = max(final_results, key=lambda x: final_results[x]["F1"])
best_model = final_results[best_model_name]["model"]
best_threshold = final_results[best_model_name]["Threshold"]

print("\nFINAL MODEL:", best_model_name)
print("Threshold:", best_threshold)

In [ ]:
import joblib
from pathlib import Path

out_dir = Path("../../models/diabetes")
out_dir.mkdir(parents=True, exist_ok=True)

joblib.dump(best_model, out_dir / "model.pkl")
joblib.dump(scaler, out_dir / "scaler.pkl")
joblib.dump(best_threshold, out_dir / "threshold.pkl")

print("Model saved successfully")


In [ ]:
sample = X_test.iloc[0:1]

# Convert back to DataFrame after scaling
scaled = scaler.transform(sample)
scaled_df = pd.DataFrame(scaled, columns=X_test.columns)

prob = best_model.predict_proba(scaled_df)[0][1]
pred = 1 if prob >= best_threshold else 0

print("Probability:", prob)
print("Prediction:", pred)


In [ ]:
high_risk_input = pd.DataFrame(0, index=[0], columns=X_train.columns)

# numerical (match EXACT values)
high_risk_input["age"] = 16
high_risk_input["bmi"] = 17.61
high_risk_input["HbA1c_level"] = 6.5
high_risk_input["blood_glucose_level"] = 158

# binary
high_risk_input["hypertension"] = 0
high_risk_input["heart_disease"] = 0

# one-hot (IMPORTANT)
high_risk_input["gender_Male"] = 1
high_risk_input["smoking_history_never"] = 1

# predict
prob = best_model.predict_proba(high_risk_input)[0][1]
pred = 1 if prob >= best_threshold else 0

print("Probability:", prob)
print("Prediction:", pred)


In [ ]:
import random

idx = random.randint(0, len(X) - 1)

row = X.iloc[idx]
actual = y.iloc[idx]

prob = best_model.predict_proba(row.to_frame().T)[0][1]
pred = 1 if prob >= best_threshold else 0

print("INDEX:", idx)
print("
MODEL INPUT (one-hot):")
print(row)

print("
ACTUAL:", actual)
print("PREDICTED:", pred)
print("PROB:", prob)
